In [ ]:
%pip install --user -q -U transformers accelerate datasets pillow tqdm


# LLaVA debiased VQAv2 samples

Combined sample generation and configurable-layer debiased provenance extraction for `llava-hf/llava-1.5-7b-hf`.


In [ ]:
from pathlib import Path
import json

import torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoProcessor, LlavaForConditionalGeneration


In [ ]:
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()
    if HERE.name != "llava" and (HERE / "llava").exists():
        HERE = HERE / "llava"

MODEL_NAME = "llava-hf/llava-1.5-7b-hf"
MODEL_TAG = MODEL_NAME.rsplit("/", 1)[-1].replace("-", "_")
SAMPLES_PATH = HERE / "vqav2_samples" / "samples.jsonl"
NUM_SAMPLES = 20
MAX_NEW_TOKENS = 32
TARGET_LAYER = 10
OUTPUT_DIR = HERE / f"{MODEL_TAG}_layer{TARGET_LAYER}_debiased_outputs"

ALPHA = 0.7
BETA = 0.5
FLOOR = 1e-4

OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
def load_target_samples():
    if not SAMPLES_PATH.exists():
        raise FileNotFoundError(f"Missing {SAMPLES_PATH}")

    samples = []
    with open(SAMPLES_PATH, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f, 1):
            if not line.strip():
                continue
            sample = json.loads(line)
            sample["sample_id"] = int(sample.get("sample_id", idx))
            sample["question_id"] = int(sample["question_id"])
            sample["image_id"] = int(sample["image_id"])

            image_path = Path(sample["image_path"])
            if not image_path.is_absolute():
                image_path = HERE / image_path
            sample["image_path"] = image_path

            samples.append(sample)
            if len(samples) >= NUM_SAMPLES:
                break

    if not samples:
        raise RuntimeError(f"No samples found in {SAMPLES_PATH}")

    print(f"Loaded {len(samples)} extracted VQAv2 samples")
    return samples


samples = load_target_samples()


In [ ]:
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="auto",
    attn_implementation="eager",
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)
tokenizer = processor.tokenizer
model_device = next(model.parameters()).device

print(torch.__version__, torch.version.cuda, torch.cuda.is_available())


In [ ]:
def as_pil_image(image_path):
    return Image.open(image_path).convert("RGB")


def build_prompt(question):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question},
            ],
        }
    ]
    try:
        return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        return f"USER: <image>\n{question}\nASSISTANT:"


def move_inputs(inputs):
    moved = {}
    for key, value in inputs.items():
        if not hasattr(value, "to"):
            moved[key] = value
        elif torch.is_floating_point(value):
            moved[key] = value.to(device=model_device, dtype=model.dtype)
        else:
            moved[key] = value.to(model_device)
    return moved


def make_inputs(sample):
    image = as_pil_image(sample["image_path"])
    prompt = build_prompt(sample["question"])
    inputs = processor(text=prompt, images=image, return_tensors="pt")
    return move_inputs(inputs), image.size


def image_token_id():
    token_id = getattr(model.config, "image_token_index", None)
    if token_id is None:
        token_id = getattr(model.config, "image_token_id", None)
    if token_id is None:
        token_id = tokenizer.convert_tokens_to_ids("<image>")
    if token_id is None or token_id < 0:
        raise ValueError("Could not resolve LLaVA image token id")
    return int(token_id)


IMAGE_TOKEN_ID = image_token_id()


In [ ]:
CHAT_ROLE_TOKEN_SEQUENCES = [
    ["\u2581US", "ER", ":"],
    ["US", "ER", ":"],
    ["\u2581USER", ":"],
    ["USER", ":"],
    ["\u2581A", "SS", "IST", "ANT", ":"],
    ["A", "SS", "IST", "ANT", ":"],
    ["\u2581ASS", "IST", "ANT", ":"],
    ["ASS", "IST", "ANT", ":"],
    ["\u2581ASSISTANT", ":"],
    ["ASSISTANT", ":"],
]
CHAT_TEMPLATE_TEXT_TOKENS = {"\u2581", "<0x0A>"}


def remove_chat_template_tokens(tokens, text_mask):
    text_mask = text_mask.clone()
    for i, token in enumerate(tokens):
        if token in CHAT_TEMPLATE_TEXT_TOKENS:
            text_mask[i] = False

    for sequence in CHAT_ROLE_TOKEN_SEQUENCES:
        width = len(sequence)
        for start in range(0, len(tokens) - width + 1):
            if tokens[start : start + width] == sequence:
                text_mask[start : start + width] = False
    return text_mask


def token_masks(input_ids):
    input_ids = input_ids.detach().cpu()
    tokens = tokenizer.convert_ids_to_tokens(input_ids.tolist())
    image = input_ids == IMAGE_TOKEN_ID
    text = torch.ones_like(image, dtype=torch.bool)

    for special_id in tokenizer.all_special_ids:
        text &= input_ids != special_id
    text &= ~image
    text = remove_chat_template_tokens(tokens, text)

    prompt = text | image
    return text, image, prompt

def normalize_rows(x, eps=1e-12):
    x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    row_sum = x.sum(-1, keepdim=True)
    return torch.where(row_sum > eps, x / row_sum.clamp_min(eps), torch.zeros_like(x))


def compute_layer_debiased(attentions, input_ids, alpha=ALPHA, beta=BETA, floor=FLOOR):
    if TARGET_LAYER >= len(attentions):
        raise ValueError(f"TARGET_LAYER={TARGET_LAYER} but model returned {len(attentions)} layers")

    text_tokens, image_tokens, prompt_tokens = token_masks(input_ids)
    device = attentions[0].device
    dtype = torch.float32

    prompt_tokens = prompt_tokens.to(device)
    image_tokens = image_tokens.to(device)
    text_tokens = text_tokens.to(device)

    n_prompt = int(prompt_tokens.sum().item())
    n_image = int(image_tokens.sum().item())
    n_text = int(text_tokens.sum().item())
    if n_image == 0:
        raise ValueError("No image tokens found in the LLaVA prompt")
    if n_text == 0:
        raise ValueError("No text tokens found in the LLaVA prompt")

    provenance = torch.zeros(n_prompt, n_image, dtype=dtype, device=device)
    provenance[image_tokens[prompt_tokens]] = torch.eye(n_image, dtype=dtype, device=device)
    baseline_provenance = provenance.clone()

    prompt_idx = torch.where(prompt_tokens)[0].to(device)
    text_idx = torch.where(text_tokens)[0].to(device)

    prompt_reach = (prompt_idx[:, None] >= prompt_idx[None, :]).to(dtype)
    prompt_reach = normalize_rows(prompt_reach)

    text_reach = (text_idx[:, None] >= prompt_idx[None, :]).to(dtype)
    text_reach = normalize_rows(text_reach)

    identity = torch.eye(n_prompt, dtype=dtype, device=device)

    for layer in range(TARGET_LAYER + 1):
        if layer:
            attn = attentions[layer - 1][0, :, prompt_tokens][:, :, prompt_tokens].mean(0).to(dtype)
            attn = normalize_rows(attn)
            provenance = (beta * identity + (1 - beta) * attn) @ provenance
            provenance = torch.nan_to_num(provenance, nan=0.0, posinf=0.0, neginf=0.0)
            baseline_provenance = (beta * identity + (1 - beta) * prompt_reach) @ baseline_provenance

    text_attn = attentions[TARGET_LAYER][0, :, text_tokens][:, :, prompt_tokens].mean(0).to(dtype)
    text_attn = normalize_rows(text_attn)

    actual = text_attn @ provenance
    baseline = text_reach @ baseline_provenance
    debiased = actual / baseline.clamp_min(floor).pow(alpha)
    debiased = normalize_rows(debiased)

    return {
        "baseline": baseline.detach().cpu(),
        "debiased": debiased.detach().cpu(),
        "debiased_mean": debiased.mean(0).detach().cpu(),
        "text_mask": text_tokens.detach().cpu(),
        "image_mask": image_tokens.detach().cpu(),
        "prompt_mask": prompt_tokens.detach().cpu(),
    }


In [ ]:
def collect_sample(sample):
    inputs, image_size = make_inputs(sample)
    input_ids = inputs["input_ids"][0]

    with torch.no_grad():
        generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        outputs = model(**inputs, output_attentions=True, return_dict=True)

    answer_ids = generated[0, input_ids.shape[0] :]
    answer = processor.decode(answer_ids, skip_special_tokens=True).strip()
    layer_bundle = compute_layer_debiased(outputs.attentions, input_ids)

    tokens = tokenizer.convert_ids_to_tokens(input_ids.detach().cpu().tolist())
    out = {
        "sample_id": sample["sample_id"],
        "question_id": int(sample["question_id"]),
        "image_id": int(sample["image_id"]),
        "question": sample["question"],
        "answer": answer,
        "ground_truth_answer": sample.get("ground_truth_answer"),
        "vqa_answers": sample.get("vqa_answers"),
        "model_name": MODEL_NAME,
        "target_layer": TARGET_LAYER,
        "alpha": ALPHA,
        "beta": BETA,
        "floor": FLOOR,
        "image_size": image_size,
        "num_image_tokens": int(layer_bundle["image_mask"].sum().item()),
        "num_text_tokens": int(layer_bundle["text_mask"].sum().item()),
        "input_ids": input_ids.detach().cpu(),
        "tokens": tokens,
        **layer_bundle,
    }

    path = OUTPUT_DIR / f"sample_{sample['sample_id']}.pt"
    torch.save(out, path)
    print(sample["sample_id"], "tokens=", out["num_image_tokens"], "answer=", answer, "->", path)
    return {
        "sample_id": out["sample_id"],
        "question_id": out["question_id"],
        "image_id": out["image_id"],
        "question": out["question"],
        "answer": out["answer"],
        "ground_truth_answer": out["ground_truth_answer"],
        "output_path": str(path),
        "shape": list(out["debiased"].shape),
        "num_image_tokens": out["num_image_tokens"],
        "num_text_tokens": out["num_text_tokens"],
    }


In [ ]:
records = []
for sample in samples:
    records.append(collect_sample(sample))

manifest = {
    "model_name": MODEL_NAME,
    "samples_path": str(SAMPLES_PATH),
    "num_samples": len(records),
    "target_layer": TARGET_LAYER,
    "alpha": ALPHA,
    "beta": BETA,
    "floor": FLOOR,
    "samples": records,
}

manifest_path = OUTPUT_DIR / "manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("wrote", manifest_path)


In [ ]:
def clean_token(token):
    return token.replace(chr(0x0120), "").replace(chr(0x2581), "").replace(chr(0x010A), "\\n")


def plot_importances(sample_path, text_token_indices=None, columns=4, cmap="viridis", overlay_image=False):
    obj = torch.load(sample_path, map_location="cpu")
    debiased = obj["debiased"].float()
    mean_scores = obj.get("debiased_mean", debiased.mean(0)).float()
    num_image_tokens = int(mean_scores.numel())
    grid_size = int(num_image_tokens ** 0.5)
    if grid_size * grid_size != num_image_tokens:
        raise ValueError(f"Expected a square image-token grid, got {num_image_tokens} tokens")

    tokens = obj.get("tokens", [])
    text_mask = obj.get("text_mask")
    text_positions = text_mask.nonzero().flatten().tolist() if text_mask is not None else []

    if text_token_indices is None:
        scores = mean_scores.unsqueeze(0)
        titles = [f"sample {obj['sample_id']} mean layer {obj['target_layer']}"]
        columns = 1
    else:
        if isinstance(text_token_indices, int):
            text_token_indices = [text_token_indices]
        scores = debiased[text_token_indices]
        titles = []
        for idx in text_token_indices:
            if idx < len(text_positions) and text_positions[idx] < len(tokens):
                titles.append(f"{idx}: {clean_token(tokens[text_positions[idx]])}")
            else:
                titles.append(str(idx))

    scores = scores.nan_to_num(0)
    print(
        "scores:",
        "shape=", tuple(scores.shape),
        "min=", float(scores.min()),
        "max=", float(scores.max()),
        "sum=", float(scores.sum()),
    )

    rows = (scores.shape[0] + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 4 * rows), squeeze=False)
    axes = axes.flatten()

    if overlay_image and "image_path" in obj:
        image_path = Path(obj["image_path"])
        if not image_path.is_absolute():
            image_path = HERE / image_path
        if image_path.exists():
            image = Image.open(image_path).convert("RGB")
        else:
            image = None
    else:
        image = None

    vmin = float(scores.min())
    vmax = float(scores.max())
    if vmax <= vmin:
        vmax = vmin + 1e-12

    im = None
    for i, ax in enumerate(axes):
        ax.axis("off")
        if i >= scores.shape[0]:
            continue
        heatmap = scores[i].reshape(grid_size, grid_size)
        if image is not None:
            ax.imshow(image)
            im = ax.imshow(heatmap, cmap=cmap, alpha=0.55, extent=(0, image.width, image.height, 0), vmin=vmin, vmax=vmax)
        else:
            im = ax.imshow(heatmap, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
        ax.set_title(titles[i])

    for ax in axes[scores.shape[0]:]:
        ax.set_visible(False)

    if im is not None:
        fig.colorbar(im, ax=axes[: scores.shape[0]].tolist(), label="Debiased importance", fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


sample_path = OUTPUT_DIR / "sample_1.pt"
plot_importances(sample_path)
# Optional overlay, if the saved bundle includes image_path:
# plot_importances(sample_path, overlay_image=True)
# Per-text-token view:
# plot_importances(sample_path, text_token_indices=list(range(8)), columns=4)
